# Minimum variance with cvxpy

This notebook solves a long-only, fully invested minimum-variance problem. The demos cap each name at 5%. The default `highst` in `cp_weight` is 1; the second solve passes `0.05`. The estimation window is 2021-09-01 through 2026-09-30, and the universe is the 50 largest index weights on 2026-09-30 among names missing no more than 5% of that window. Days with any missing return are dropped before the covariance is formed.


# 用 cvxpy 做最小方差

这个笔记本求解只做多、权重和为 1 的最小方差。演示时把单票上限设为 5%。`cp_weight` 的默认上限 `highst` 是 1，第二问在调用时传入 `0.05`。估计区间是 2021-09-01 到 2026-09-30，股票是 2026-09-30 指数权重最大、且区间内缺失不超过 5% 的 50 只。求协方差之前先丢掉任何有缺失的交易日。


In [1]:
import pandas as pd
import numpy as np
import cvxpy as cp
import edhec_risk_kit as erk


## Universe


## 股票池


In [2]:
snapshot = erk.read_hs300_weights().iloc[-2].dropna().sort_values(ascending=False)
raw = erk.read_returns().loc["2021-09-01":"2026-09-30"]
eligible = raw.loc[:, raw.isna().mean() <= 0.05]
codes = snapshot.index[snapshot.index.isin(eligible.columns)][:50]
history = eligible[codes].dropna(how="any")
cov = history.cov()
print(snapshot.name.date(), history.shape[0], history.shape[1])


2026-09-30 1153 50


## Minimum variance with a 5% cap

Minimize `w'Σw` subject to weights summing to 1 and `0 ≤ w ≤ 0.05`. The 5% bound is in the constraint list passed to the solver. The cell prints the solver status, the sum of weights, the largest weight, and how many names sit against the cap.

The largest weight in this solve is 0.05, and 15 names sit on the cap.


## 带 5% 上限的最小方差

最小化 `w'Σw`。约束是权重和为 1，并且 `0 ≤ w ≤ 0.05`。`0.05` 写在求解使用的约束列表里。下面打印求解状态、权重和、最大权重，以及贴近上限的股票数量。

这一问的最大权重是 0.05，有 15 只股票顶在上限上。


In [3]:
n_names = cov.shape[0]
weight = cp.Variable(n_names)
constraints = [cp.sum(weight) == 1, weight >= 0, weight <= 0.05]
problem = cp.Problem(cp.Minimize(cp.quad_form(weight, cov.to_numpy())), constraints)
problem.solve()
minimum_variance = pd.Series(weight.value, index=cov.index)
print(problem.status)
print(float(minimum_variance.sum()), float(minimum_variance.max()), int((minimum_variance >= 0.05 - 1e-4).sum()))
minimum_variance.sort_values(ascending=False).head(10)


optimal
1.0 0.05 15


600519.SH    0.05
600900.SH    0.05
601398.SH    0.05
601166.SH    0.05
601328.SH    0.05
600919.SH    0.05
000651.SZ    0.05
002415.SZ    0.05
600000.SH    0.05
601088.SH    0.05
dtype: float64

## `cp_weight` with a return target

`cp_weight` minimizes `w'Σw + λ||w − w_old||_1 + 1000 * slack` with `slack ≥ 0`. The return constraint uses the in-sample mean daily return: that mean dotted with the weights, plus slack, must be at least `0.10/252` (about 0.000397). This call passes `highst=0.05`. `λ` stays at its default of 0 and `w_old` stays at its default of 0, so the turnover term is zero and does not change the weights.

The capped minimum-variance portfolio from the previous section already has an in-sample daily mean of about 0.000413, above that target. The return constraint does not bind, slack stays at 0, and the weights below match the previous portfolio up to solver tolerance. The penalty per unit of slack is 1000, and this portfolio's daily variance is about `6e-5`. Slack is unused here because the target is already met. The printed mean is still the in-sample daily mean on this estimation window.

The function solves the problem with CLARABEL. Default OSQP tolerances stop at a feasible weight whose variance is about twice as large.


## 带收益目标的 `cp_weight`

`cp_weight` 最小化 `w'Σw + λ||w − w_old||_1 + 1000 * slack`，其中 `slack ≥ 0`。收益约束用样本内的日均收益：日均收益与权重的内积，加上 slack，不低于 `0.10/252`（大约 0.000397）。这一问传入 `highst=0.05`。`λ` 用默认值 0，`w_old` 用默认值 0，所以换手项是 0，不改变权重。

上一问那个带 5% 上限的最小方差组合，样本内日均收益大约是 0.000413，已经高于这个目标。收益约束没有被卡住，slack 为 0，下面的权重和上一问几乎相同，剩下的差别是求解器误差。slack 的单位惩罚是 1000，这一问的日方差大约是 `6e-5`。这次用不到 slack，是因为目标本身已经满足。打印出来的平均收益仍是这段估计样本里的日均收益。

这个函数用 CLARABEL 求解。同一个问题如果交给默认精度的 OSQP，会停在一个满足约束、但方差大约高一倍的权重上。


In [4]:
targeted = pd.Series(
    erk.cp_weight(cov.to_numpy(), mu=history.to_numpy(), target=0.1, highst=0.05),
    index=cov.index,
)
in_sample = history.to_numpy() @ targeted.to_numpy()
print(float(targeted.sum()), float(targeted.max()), float(targeted.min()))
print(int((targeted >= 0.05 - 1e-4).sum()))
print(float(in_sample.mean()), 0.1 / 252)
targeted.sort_values(ascending=False).head(10)


0.9999999999999997 0.04999937886385706 5.459085070281147e-07
15
0.00041297120366952947 0.0003968253968253968


601398.SH    0.049999
600900.SH    0.049999
601288.SH    0.049999
601328.SH    0.049999
600000.SH    0.049999
601816.SH    0.049999
000651.SZ    0.049998
600887.SH    0.049998
601088.SH    0.049998
000725.SZ    0.049998
dtype: float64